# 🥈 Silver — Yellow Taxi cleaning & enrichment

**Purpose.** Transform the Bronze Yellow Taxi data into a clean, typed, enriched table that downstream Gold aggregations can trust.

**Source:** `workspace.bronze.yellow_taxi` (5.97M rows)
**Output:** `workspace.silver.yellow_taxi`

**Transformations applied:**
- Drop rows with implausible values (negative fares, zero distance, dropoff before pickup)
- Enforce sensible types and column casing
- Enrich with the TLC taxi-zone lookup — replace opaque `PULocationID` / `DOLocationID` with named `pickup_borough` / `pickup_zone` / `dropoff_borough` / `dropoff_zone`
- Derive convenience columns: `trip_duration_minutes`, `avg_speed_mph`

**Quality contract:** Every row in Silver has positive fare, positive distance, pickup < dropoff, and a valid pickup + dropoff zone.

In [0]:
# Configuration -----------------------------------------------------------------
SOURCE_TABLE = "workspace.bronze.yellow_taxi"
TARGET_CATALOG = "workspace"
TARGET_SCHEMA = "silver"
TARGET_TABLE = "yellow_taxi"
FULL_TABLE_NAME = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

# Volume to stage the zone lookup CSV (same pattern as Bronze).
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
ZONE_VOLUME = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.reference"
ZONE_VOLUME_PATH = f"/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/reference"
ZONE_LOCAL_PATH = f"{ZONE_VOLUME_PATH}/taxi_zone_lookup.csv"

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {FULL_TABLE_NAME}")
print(f"Zone lookup will land at: {ZONE_LOCAL_PATH}")

In [0]:
# Ensure Silver schema exists.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
spark.sql(f"USE {TARGET_CATALOG}.{TARGET_SCHEMA}")

# Volume for reference data (zone lookup, future small CSVs).
spark.sql(f"CREATE VOLUME IF NOT EXISTS {ZONE_VOLUME}")

print(f"✓ Schema {TARGET_CATALOG}.{TARGET_SCHEMA} and Volume reference are ready.")

In [0]:
import os
import urllib.request

if os.path.exists(ZONE_LOCAL_PATH):
    size_kb = os.path.getsize(ZONE_LOCAL_PATH) / 1024
    print(f"✓ already in volume ({size_kb:.1f} KB): {ZONE_LOCAL_PATH}")
else:
    print(f"↓ downloading: {ZONE_LOOKUP_URL}")
    urllib.request.urlretrieve(ZONE_LOOKUP_URL, ZONE_LOCAL_PATH)
    size_kb = os.path.getsize(ZONE_LOCAL_PATH) / 1024
    print(f"  saved {size_kb:.1f} KB → {ZONE_LOCAL_PATH}")

In [0]:
zones_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ZONE_LOCAL_PATH)
)

print(f"Zone lookup: {zones_df.count()} rows, {len(zones_df.columns)} columns")
zones_df.printSchema()
display(zones_df.limit(5))

In [0]:
from pyspark.sql.functions import col, unix_timestamp, round as spark_round, when

# Read the Bronze table.
bronze_df = spark.table(SOURCE_TABLE)
bronze_count = bronze_df.count()
print(f"Bronze input: {bronze_count:,} rows")

# Apply quality filters. The data has well-known issues:
#   - Negative or zero fares (data-entry errors)
#   - Zero-distance trips (cancelled rides recorded as completed)
#   - Dropoff timestamp before pickup (timezone or clock bugs)
#   - Trips longer than 24h (almost certainly errors or fare disputes)
#   - Implausible passenger counts
#
# We filter conservatively: only drop rows we're confident are bad.
cleaned_df = (
    bronze_df
    .filter(col("fare_amount") > 0)
    .filter(col("trip_distance") > 0)
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    .filter(
        unix_timestamp("tpep_dropoff_datetime")
        - unix_timestamp("tpep_pickup_datetime")
        <= 24 * 60 * 60
    )
    .filter(col("passenger_count").between(1, 6))
    # Belt-and-braces: also drop rows missing pickup or dropoff zone id
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
)

cleaned_count = cleaned_df.count()
dropped = bronze_count - cleaned_count
drop_pct = dropped / bronze_count * 100
print(f"After quality filters: {cleaned_count:,} rows")
print(f"Dropped: {dropped:,} rows ({drop_pct:.2f}%)")

In [0]:
# Prepare two aliased copies of the zone lookup for the two joins
# (pickup + dropoff). Using aliases avoids ambiguous column names after join.
pickup_zones = (
    zones_df
    .select(
        col("LocationID").alias("PULocationID"),
        col("Borough").alias("pickup_borough"),
        col("Zone").alias("pickup_zone"),
        col("service_zone").alias("pickup_service_zone"),
    )
)

dropoff_zones = (
    zones_df
    .select(
        col("LocationID").alias("DOLocationID"),
        col("Borough").alias("dropoff_borough"),
        col("Zone").alias("dropoff_zone"),
        col("service_zone").alias("dropoff_service_zone"),
    )
)

# Two left joins (preserve all cleaned rows even if a zone is missing).
enriched_df = (
    cleaned_df
    .join(pickup_zones, on="PULocationID", how="left")
    .join(dropoff_zones, on="DOLocationID", how="left")
)

# Derived analytical columns.
trip_seconds = (
    unix_timestamp("tpep_dropoff_datetime")
    - unix_timestamp("tpep_pickup_datetime")
)
enriched_df = (
    enriched_df
    .withColumn("trip_duration_minutes", spark_round(trip_seconds / 60.0, 2))
    .withColumn(
        "avg_speed_mph",
        when(trip_seconds > 0, spark_round(col("trip_distance") / (trip_seconds / 3600.0), 2))
        .otherwise(None),
    )
)

print(f"Enriched schema ({len(enriched_df.columns)} columns):")
enriched_df.printSchema()

# Sanity-check: how many rows didn't get a pickup zone match?
unmatched_pickup = enriched_df.filter(col("pickup_zone").isNull()).count()
unmatched_dropoff = enriched_df.filter(col("dropoff_zone").isNull()).count()
print(f"\nUnmatched pickup zone:  {unmatched_pickup:,} rows")
print(f"Unmatched dropoff zone: {unmatched_dropoff:,} rows")

In [0]:
print("Sample enriched rows:")
display(
    enriched_df.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_minutes",
        "trip_distance",
        "avg_speed_mph",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
        "fare_amount",
        "total_amount",
    ).limit(10)
)

# Quick distribution: trips per pickup borough — sanity check that joins worked.
print("\nTrips per pickup borough:")
display(
    enriched_df
    .groupBy("pickup_borough")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
(
    enriched_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

silver_count = spark.table(FULL_TABLE_NAME).count()
print(f"✓ Wrote {silver_count:,} rows to {FULL_TABLE_NAME}")

In [0]:
print("Silver table: a real working sample\n")
display(
    spark.table(FULL_TABLE_NAME)
    .select(
        "tpep_pickup_datetime",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "trip_duration_minutes",
        "avg_speed_mph",
        "fare_amount",
        "total_amount",
    )
    .limit(5)
)

print("\nTrips per pickup borough (sanity-check the join worked):")
display(
    spark.table(FULL_TABLE_NAME)
    .groupBy("pickup_borough")
    .count()
    .orderBy("count", ascending=False)
)

print("\nTrip-duration distribution (rough sanity-check the derived columns are sensible):")
spark.table(FULL_TABLE_NAME).selectExpr(
    "min(trip_duration_minutes) as min_min",
    "max(trip_duration_minutes) as max_min",
    "round(avg(trip_duration_minutes), 2) as avg_min",
    "round(percentile_approx(trip_duration_minutes, 0.5), 2) as median_min",
).show()